In [1]:
import sys
!pip install google-search-results --quiet
!pip install google-genai --quiet

# Primero, instala Pillow a una versión que debería ser compatible con torchvision 0.25.0
# y otras librerías como scikit-image (que pide >= 10.1).
# Pillow 10.1.0 parece una buena candidata. Esto generará una advertencia para rembg, pero debería resolver el ImportError.
!pip install Pillow==10.1.0

# Ahora instala las librerías principales. Deberían respetar la versión de Pillow ya instalada.
!pip install torch transformers google-generativeai langdetect --quiet
!pip install torchvision --quiet # Instala torchvision explícitamente
!pip install huggingface_hub
!pip install -q gradio_client

!pip install qwen-vl-utils -q

!pip install "rembg[cpu]" onnxruntime -q
!apt-get install -y fonts-dejavu -q

!pip install beautifulsoup4 -q

# Reiniciar el entorno de ejecución (Runtime) después de este cambio y ejecutar todo de nuevo.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.
Reading package lists...
Building dependency tree...
Reading state information...
fonts-dejavu is already the newest version (2.37-2build1).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.


In [2]:
import json
import re
import time
import torch
import google.generativeai as genai
from transformers import pipeline
from google.colab import userdata
from serpapi import GoogleSearch
import requests
import os

from transformers import WhisperProcessor, WhisperForConditionalGeneration
from IPython.display import display, Image as IPImage


from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/MIARMARIO/')

from Modulo1 import *
from Modulo30 import *
from Modulo2 import *
from Modulo40 import *


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
whisper = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3",torch_dtype=torch.float32 )
device = "cuda" if torch.cuda.is_available() else "cpu"
whisper_model = whisper.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [4]:
GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')   # Google Cloud → Custom Search API
SEARCH_ENGINE_ID = userdata.get('SEARCH_ENGINE_ID') # programmablesearchengine.google.com → cx

SERPAPI_KEY = userdata.get('SERPAPI_KEY')  # serpapi.com → dashboard → API Key

CARPETA_ROPA      = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO"          # Carpeta con fotos del armario
RUTA_ARMARIO_JSON = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/armario.json"  # BD persistente de prendas

CARPETA_SALIDA    = '/content/drive/MyDrive/Colab Notebooks/MIARMARIO/CARPETA SALIDA'

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

# Paso 1. Cargar armario.

In [5]:
'''if forzar_reconstruccion and os.path.exists(RUTA_ARMARIO_JSON):
        os.remove(RUTA_ARMARIO_JSON)
        print("[Paso 1] Armario eliminado para reconstrucción completa")'''

if os.path.exists(RUTA_ARMARIO_JSON):
    print("[Paso 1] Usando armario.json existente.")
    armario = cargar_armario(RUTA_ARMARIO_JSON)
elif os.path.exists(CARPETA_ROPA):
    print("[Paso 1] Construyendo armario desde las fotos...")
    armario = modulo2_construir_armario(
        carpeta_ropa=CARPETA_ROPA,
        ruta_armario_json=RUTA_ARMARIO_JSON,
    )
else:
    raise FileNotFoundError("No hay armario.json ni carpeta ARMARIO. Añade fotos para continuar.")


[Paso 1] Usando armario.json existente.


In [6]:
print(json.dumps(armario, ensure_ascii=False, indent=2))

[
  {
    "tipo": "chaqueta",
    "color": "azul",
    "formalidad": "casual",
    "temporada": [
      "invierno"
    ],
    "id": "prenda_001_1",
    "imagen_path": "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO/abrigo.jpg"
  },
  {
    "tipo": "pantalón",
    "color": "negro",
    "formalidad": "casual",
    "temporada": [
      "invierno",
      "primavera"
    ],
    "id": "prenda_002_1",
    "imagen_path": "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO/pantalontraje.png"
  },
  {
    "tipo": "camisa",
    "color": "gris",
    "estampado": "liso",
    "formalidad": "formal",
    "temporada": [
      "primavera",
      "verano"
    ],
    "id": "prenda_003_1",
    "imagen_path": "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO/polo.webp"
  },
  {
    "tipo": "camisa",
    "color": "azul",
    "estampado": "liso",
    "formalidad": "formal",
    "temporada": [
      "primavera",
      "verano"
    ],
    "id": "prenda_004_1",
    "imagen_path": "/conte

# Paso 2. Procesar instrucciones del ususario.


In [ ]:
resultado_mod1 = modulo1_entrada(
    fuente="texto",
    texto="hola quiero que me digas un outfit que sea bueno para irme de escalada por el dia que hace calor",
    ciudad="Cabra",
    model=model
)



Límite alcanzado, esperando 35s...


Límite alcanzado, esperando 70s...


In [7]:
#a mano por si no quedan consultas
resultado_mod1 = {
  "texto": "Quiero un outfit para una cena esta noche. Mi jefe viene, así que necesito ir arreglado.",
  "fuente": "texto",
  "idioma": "es",
  "contexto": {
    "ciudad": "Sevilla",
    "temperatura": 19.2,
    "unidad": "celsius"
  }
}


In [8]:
print(json.dumps(resultado_mod1, ensure_ascii=False, indent=2))

{
  "texto": "Busco un outfit para un día de escalada con calor.",
  "fuente": "texto",
  "idioma": "es",
  "contexto": {
    "ciudad": "Cabra",
    "temperatura": 15.7,
    "unidad": "celsius"
  }
}


In [9]:
# Ejemplo 2 — Entrada por voz
resultado_mod1 = modulo1_entrada(
    fuente="voz",
    ruta_audio="/content/drive/MyDrive/Colab Notebooks/MIARMARIO/mi_audio.mp4",# <-- cambia por tu archivo
    ciudad="Sevilla",
    model=model,
    whisper=whisper,
    processor=processor,
    device=device
)



Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

In [9]:
print(json.dumps(resultado_mod1, ensure_ascii=False, indent=2))

{
  "texto": "Busco un outfit para un día de escalada con calor.",
  "fuente": "texto",
  "idioma": "es",
  "contexto": {
    "ciudad": "Cabra",
    "temperatura": 15.7,
    "unidad": "celsius"
  }
}


# Paso 3. Seleccionar outfit.

In [10]:
'''armario = {
    "armario": [
        {
            "id": "prenda_001",
            "tipo": "camisa",
            "color": "blanco",
            "estampado": "liso",
            "material": "algodón",
            "formalidad": "formal",
            "temporada": ["primavera", "verano", "otoño"],
            "imagen_path": "fotos/camisa_blanca_001.jpg"
        },
        {
            "id": "prenda_002",
            "tipo": "pantalón",
            "color": "negro",
            "estampado": "liso",
            "material": "lana",
            "formalidad": "formal",
            "temporada": ["otoño", "invierno"],
            "imagen_path": "fotos/pantalon_negro_002.jpg"
        }
    ]
}'''


resultado_mod3 = modulo3_seleccion_outfit(
    armario_json=armario,
    instruccion_usuario=resultado_mod1["texto"],
    contexto={"temperatura": 18, "ocasion": "cena"},
    model=model,
    SERPAPI_KEY=SERPAPI_KEY
)



Límite alcanzado, esperando 35s...


Límite alcanzado, esperando 70s...


Límite alcanzado, esperando 105s...


Exception: Se agotaron los reintentos

In [12]:
print(json.dumps(resultado_mod3, ensure_ascii=False, indent=2))

{
  "outfit": {
    "prendas": [
      {
        "id": "prenda_010_1",
        "tipo": "camiseta",
        "color": "verde"
      },
      {
        "id": "prenda_018_1",
        "tipo": "shorts",
        "color": "gris"
      },
      {
        "id": "prenda_012_1",
        "tipo": "zapatos",
        "color": "rojo"
      }
    ],
    "estilo": "sport",
    "ocasion": "cena",
    "temperatura": 18
  },
  "prompt_imagen": "A man wearing a green athletic t-shirt, grey athletic shorts, and red Nike sneakers. He is also wearing a black technical sports cap. The overall style is sporty and ready for climbing. Full body, fashion photography, studio lighting, dynamic pose.",
  "referencias_tiendas": [
    {
      "prenda": "Pantalones Cortos Deportivos y de Gimnasio para Hombre",
      "tienda": "https://www.asos.com › Inicio › Hombre › Activewear",
      "enlace": "https://www.asos.com/es/hombre/ropa-deportiva/pantalones-cortos/cat/?cid=27178",
      "imagen": "",
      "por_que": "Ofrece u

In [22]:
#resultado por si no quedan consultas

resultado_mod3 = {
  "outfit": {
    "prendas": [
      {
        "id": "prenda_004_1",
        "tipo": "camisa",
        "color": "azul"
      },
      {
        "id": "prenda_002_1",
        "tipo": "pantalón",
        "color": "negro"
      },
      {
        "id": "prenda_008_1",
        "tipo": "zapatos",
        "color": "marrón"
      },
      {
        "id": "prenda_019_1",
        "tipo": "cinturón",
        "color": "marrón oscuro"
      },
      {
        "id": "prenda_021_1",
        "tipo": "reloj",
        "color": "negro"
      }
    ],
    "estilo": "formal",
    "ocasion": "cena",
    "temperatura": 18
  },
  "prompt_imagen": "A man wearing a formal plain blue button-up shirt, black formal trousers, dark brown leather penny loafers, a dark brown plain leather belt, and a minimalist black watch. Full body, fashion photography, studio lighting, elegant, professional, confident pose.",
  "referencias_tiendas": [
    {
      "prenda": "Corbatas y pajaritas de hombre en color azul",
      "tienda": "Zalando.es",
      "enlace": "https://www.zalando.es/corbatas-pajaritas-hombre/_azul/",
      "por_que": "Para añadir un toque de sofisticación a la camisa azul y elevar aún más la formalidad para una cena de empresa."
    },
    {
      "prenda": "Zapatos Formales Hombre | ZARA España",
      "tienda": "https://www.zara.com › ... › COLECCIÓN › ZAPATOS",
      "enlace": "https://www.zara.com/es/es/hombre-zapatos-formal-l2568.html",
      "por_que": "Una opción de calzado formal en negro para aquellos que prefieran un look más clásico y monocromático que los mocasines marrones."
    }
  ]
}


print(json.dumps(resultado_mod3, ensure_ascii=False, indent=2))

{
  "outfit": {
    "prendas": [
      {
        "id": "prenda_004_1",
        "tipo": "camisa",
        "color": "azul"
      },
      {
        "id": "prenda_002_1",
        "tipo": "pantalón",
        "color": "negro"
      },
      {
        "id": "prenda_008_1",
        "tipo": "zapatos",
        "color": "marrón"
      },
      {
        "id": "prenda_019_1",
        "tipo": "cinturón",
        "color": "marrón oscuro"
      },
      {
        "id": "prenda_021_1",
        "tipo": "reloj",
        "color": "negro"
      }
    ],
    "estilo": "formal",
    "ocasion": "cena",
    "temperatura": 18
  },
  "prompt_imagen": "A man wearing a formal plain blue button-up shirt, black formal trousers, dark brown leather penny loafers, a dark brown plain leather belt, and a minimalist black watch. Full body, fashion photography, studio lighting, elegant, professional, confident pose.",
  "referencias_tiendas": [
    {
      "prenda": "Corbatas y pajaritas de hombre en color azul",
      "

In [ ]:
ruta_lookbook = modulo4_generar_imagen(
    resultado_mod3 = resultado_mod3,
    armario        = armario,
    carpeta_salida = CARPETA_SALIDA,
    model          = model,
)

display(IPImage(filename=ruta_lookbook))
print(f'Lookbook guardado en: {ruta_lookbook}')

In [34]:
for p in armario:
    print(p["id"], "→", p["tipo"])

prenda_001_1 → chaqueta
prenda_002_1 → pantalón
prenda_003_1 → camisa
prenda_004_1 → camisa
prenda_005_1 → chaqueta
prenda_006_1 → camiseta
prenda_007_1 → zapatos
prenda_008_1 → zapatos
prenda_009_1 → sudadera
prenda_010_1 → camiseta
prenda_011_1 → chaqueta
prenda_012_1 → zapatos
prenda_013_1 → zapatos
prenda_014_1 → pantalón
prenda_015_1 → pantalón
prenda_016_1 → camiseta
prenda_017_1 → zapatos
prenda_018_1 → shorts
prenda_019_1 → cinturón
prenda_020_1 →  Bufanda
prenda_021_1 → reloj
prenda_022_1 → pantalón


In [35]:
# Verificar que la ruta existe y la imagen carga
prenda_test = next(p for p in armario if p["id"] == "prenda_004_1")
print(prenda_test)
print("¿Existe el archivo?", os.path.exists(prenda_test["imagen_path"]))

{'tipo': 'camisa', 'color': 'azul', 'estampado': 'liso', 'formalidad': 'formal', 'temporada': ['primavera', 'verano'], 'id': 'prenda_004_1', 'imagen_path': '/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO/camisa.jpg'}
¿Existe el archivo? True
